# Visual Question Answering (VQA) con Modelos Multimodales en Español

Este notebook guía paso a paso el proceso de entrenamiento y evaluación de un modelo multimodal basado en transformadores para la tarea de VQA en español.

## 1. Importar librerías y módulos

En esta sección se importan los módulos necesarios desde la carpeta `src` y otras librerías estándar de Python. Cada import está comentado para explicar su función.

In [1]:
# Importar configuraciones y utilidades propias del proyecto
from src.config import *  # Configuraciones generales (parámetros, rutas, seeds, etc.)
from src.data_utils import load_vqa_datasets, add_label_column  # Funciones para cargar y preparar los datos VQA
from src.model import create_multimodal_collator_and_model  # Creación del modelo multimodal y su collator
from src.train import training_loop  # Función principal de entrenamiento
from src.eval import evaluate_model  # Función para evaluar el modelo entrenado
from src.plot_utils import plot_training_statistics  # Función para graficar estadísticas de entrenamiento
from torch.optim.lr_scheduler import ReduceLROnPlateau
# Importar librerías estándar y de PyTorch
import torch  # Para manejo de tensores y dispositivos
from torch.utils.data import DataLoader, random_split  # Para crear data loaders y dividir datasets

DATASETS_PATH: /home/cvillalba/data/data3/VQA/Tesis/TesisVQA/TesisVQA/VQA-MULTI-ESP/Datasets/
IMAGES_PATH: /home/cvillalba/data/data3/VQA/Tesis/TesisVQA/TesisVQA/VQA-MULTI-ESP/Datasets/Images/images/
NEW_DATASETS_FOLDER: /home/cvillalba/data/data3/VQA/Tesis/TesisVQA/TesisVQA/VQA-MULTI-ESP/modelos/
DATASETS_PATH: /home/cvillalba/data/data3/VQA/Tesis/TesisVQA/TesisVQA/VQA-MULTI-ESP/Datasets/
IMAGES_PATH: /home/cvillalba/data/data3/VQA/Tesis/TesisVQA/TesisVQA/VQA-MULTI-ESP/Datasets/Images/images/
NEW_DATASETS_FOLDER: /home/cvillalba/data/data3/VQA/Tesis/TesisVQA/TesisVQA/VQA-MULTI-ESP/modelos/


## 2. Cargar y preparar los datos VQA

Se cargan los datasets de VQA y se añade la columna de etiquetas necesaria para el entrenamiento.  
El dataset contiene preguntas, imágenes y respuestas. La columna de etiquetas representa la clase de respuesta para cada ejemplo, lo cual es fundamental para el aprendizaje supervisado.

### Importante:
Antes de ejecutar, cambiar en config.py los paths correspondientes a los archivos a usar tanto los files de texto como las imagenes

In [2]:
# Cargar los datos VQA y el espacio de respuestas posibles
dataset, answer_space = load_vqa_datasets()

# Añadir columna de etiquetas (label) al dataset según el espacio de respuestas
dataset = add_label_column(dataset, answer_space)

# Mostrar un ejemplo del dataset para verificar el formato
print("Ejemplo de dato:", dataset['train'][0])
print("Número de respuestas posibles:", len(answer_space))

Ejemplo de dato: {'Unnamed: 0': 0, 'image_id': 'VizWiz_train_00000000.jpg', 'question': "What's the name of this product?", 'answers': "['basil leaves', 'basil leaves', 'basil', 'basil', 'basil leaves', 'basil leaves', 'basil leaves', 'basil leaves', 'basil leaves', 'basil']", 'answer_type': 'other', 'answerable': 1, 'question_trad': '¿Cuál es el nombre de este producto?', 'type': 'train', 'answers_trad': "['hojas de albahaca', 'hojas de albahaca', 'hojas de albahaca', 'hojas de albahaca', 'hojas de albahaca', 'hojas de albahaca', 'hojas de albahaca', 'hojas de albahaca', 'hojas de albahaca', 'hojas de albahaca']", 'answer_trad': 'hojas de albahaca', 'answer': 'basil leaves', 'label': 2255}
Número de respuestas posibles: 4778


In [3]:
#dataset['train']['image_id'] #== 'VizWiz_train_00017965.jpg'
#dataset['test']['image_id'] == 'VizWiz_train_00017965.jpg'


## 3. Configurar modelos y dispositivos

Se definen los modelos pre-entrenados de texto e imagen a utilizar y se configura el dispositivo de cómputo (CPU o GPU).  
El uso de GPU acelera significativamente el entrenamiento si está disponible.

In [4]:
# Definir nombres de modelos pre-entrenados (pueden ajustarse según el caso con los siguientes)
#Modelos de texto: 
#BERT='bert-base-uncased'
#BETO = 'dccuchile/bert-base-spanish-wwm-uncased'
#ROBERTA = 'bertin-project/bertin-roberta-base-spanish'

#Modelos de imagen:
#VIT = 'google/vit-base-patch16-224-in21k'
#BEIT='microsoft/beit-base-patch16-224-pt22k-ft22k'

text_model = "dccuchile/bert-base-spanish-wwm-cased"  # Modelo BERT en español
image_model = "google/vit-base-patch16-224-in21k"     # Modelo Vision Transformer

#cambiar estos nombres si se usan otros modelos
TEXT_MOLDEL_NAME = 'BERT'
IMAGE_MODEL_NAME = 'VIT'

# Seleccionar dispositivo automáticamente
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dispositivo seleccionado:", device)

Dispositivo seleccionado: cuda


## 4. Crear el modelo multimodal y el collator

Se crea el modelo multimodal y el collator, que es responsable de procesar y unir los datos de texto e imagen en batches.  
El collator asegura que cada batch esté correctamente formateado para el modelo.

In [5]:
# Crear el collator y el modelo multimodal
collator, model = create_multimodal_collator_and_model(
    answer_space=answer_space,
    text_model=text_model,
    image_model=image_model,
    device=device
)

# Mover el modelo al dispositivo seleccionado
model.to(device)

/home/cvillalba/miniconda3/envs/tesisllmenv/lib/python3.9/site-packages/transformers/models/vit/feature_extraction_vit.py:28: FutureWarning: The class ViTFeatureExtractor is deprecated and will be removed in version 5 of Transformers. Please use ViTImageProcessor instead.
  warnings.warn(
Some weights of BertModel were not initialized from the model checkpoint at dccuchile/bert-base-spanish-wwm-cased and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


MultimodalVQAModel(
  (text_encoder): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(31002, 768, padding_idx=1)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12

## 5. Preparar DataLoaders para entrenamiento y validación

Se divide el dataset en conjuntos de entrenamiento y validación.  
Se crean DataLoaders usando PyTorch, especificando el collator y el tamaño de batch.  
Los DataLoaders permiten un entrenamiento eficiente y manejo automático de batches.

In [6]:
# Definir proporción de validación
val_ratio = 0.1
train_size = int((1 - val_ratio) * len(dataset['train']))
val_size = len(dataset['train']) - train_size
print(f"Tamaño del conjunto de entrenamiento: {train_size}")
print(f"Tamaño del conjunto de validación: {val_size}")

# Dividir el conjunto de entrenamiento en train y validación
train_dataset, val_dataset = random_split(dataset['train'], [train_size, val_size])

# Definir tamaño de batch
batch_size = 32

# Crear DataLoaders
train_dataloader = DataLoader(
    train_dataset, batch_size=batch_size, shuffle=True, collate_fn=lambda batch: collator(batch, partition="train"),num_workers=2
)
val_dataloader = DataLoader(
    val_dataset, batch_size=batch_size, shuffle=False,  collate_fn=lambda batch: collator(batch, partition="val"),num_workers=2
)

Tamaño del conjunto de entrenamiento: 10835
Tamaño del conjunto de validación: 1204


In [7]:
batch = next(iter(train_dataloader))

for k,v in batch.items():
   print(k, v.shape)

batch_val = next(iter(val_dataloader))
for k,v in batch_val.items():
   print(k, v.shape)

input_ids torch.Size([32, 14])
token_type_ids torch.Size([32, 14])
attention_mask torch.Size([32, 14])
pixel_values torch.Size([32, 3, 224, 224])
labels torch.Size([32])
input_ids torch.Size([32, 24])
token_type_ids torch.Size([32, 24])
attention_mask torch.Size([32, 24])
pixel_values torch.Size([32, 3, 224, 224])
labels torch.Size([32])


## 6. Entrenar el modelo

Se entrena el modelo usando la función `training_loop`, que recibe el modelo, los data loaders, el optimizador y otros parámetros.  
La función devuelve las pérdidas y exactitudes de entrenamiento y validación por época.

- num_epochs: cantidad de veces que vamos a recorrer los datos de entrenamiento durante el entrenamiento.

- optimizer: es un objeto del paquete torch.optim, que implementa un optimizador por gradiente descendente. Lo creamos fuera del loop, pasándole por parámetro la learning rate y los parámetros del modelo que queremos optimizar

- model: es un objeto que implementa la clase abstracta torch.nn.Module, con el que se modela cualquier red neuronal o loss function.

- train_dataloader, val_dataloader: son objetos de la clase DataLoader del paquete torch.utils.data, que implementa el muestreo de los datos (armado de batches). Adentro, esas clases tienen objetos de la clase Dataset de torch.utils.data que implementan la lectura de los datos (ej. imagen y etiqueta)

In [ ]:
# Definir hiperparámetros de entrenamiento
num_epochs = 10
learning_rate = 2e-5
start_epoch=1

# Crear optimizador
# creamos una instancia de un optimizador
# pasándole los parámetros de nuestro modelo y la learning rate que queremos usar
LAMBDA = 1e-4 #L2 Regularization in PyTorch
optimizer = torch.optim.AdamW(model.parameters(), lr= learning_rate, weight_decay=LAMBDA)
is_fully_connected=False

##########################################
#agrego estas lineas para reducir el lr  when a metric has stopped improving.
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=0, verbose=True)

print('Entrenando Modelo Multimodal de VQA con Texto e Imagen')
print('text_model: ', text_model)
print('image_model: ',image_model)
print('Ejecutando con LR '+ str(learning_rate)+' y con Weight Decay ' +str(LAMBDA))

# Nombres para guardar modelos y logs
exp_name = 'VQA_Multimodal_VIZWIZ_ES_'+ TEXT_MOLDEL_NAME + '_' + IMAGE_MODEL_NAME + '_'+ str(learning_rate)+'_'+str(LAMBDA)
metrics_log = 'metrics_log_'+ exp_name +'.csv'    
best_model_chekpoint = 'best_VQA_Multimodel_VIZWIZ_ES_'+ TEXT_MOLDEL_NAME + '_' + IMAGE_MODEL_NAME + '_'+ str(learning_rate)+'_'+str(LAMBDA) +'.pt'
print('best_model_chekpoint: ', best_model_chekpoint)

#Carpeta donde se guardan los modelos
new_datasets_folder ='/home/cvillalba/data/data3/VQA/Tesis/TesisVQA/TesisVQA/VQA-MULTI-ESP/VQA_Multimodal/modelos/' 

# entrenamos la red, usando los parámetros de arriba
# Entrenar el modelo
modelo, tr_loss, val_loss, tr_acc, val_acc = training_loop(start_epoch, num_epochs, scheduler, optimizer, model, train_dataloader, val_dataloader, device, exp_name, best_model_chekpoint, new_datasets_folder, metrics_log, answer_space)

Entrenando Modelo Multimodal de VQA con Texto e Imagen
text_model:  dccuchile/bert-base-spanish-wwm-cased
image_model:  google/vit-base-patch16-224-in21k
Ejecutando con LR 2e-05 y con Weight Decay 0.0001
best_model_chekpoint:  best_VQA_Multimodel_VIZWIZ_ES_BERT_VIT_2e-05_0.0001.pt
Init time:  2025-09-07 21:24:47.287348
epoch:  1
Training...


  0%|          | 0/339 [00:00<?, ?it/s]

100%|██████████| 339/339 [06:44<00:00,  1.19s/it]


Validation...


100%|██████████| 38/38 [00:45<00:00,  1.20s/it]


...

100%|██████████| 339/339 [06:36<00:00,  1.17s/it]


Validation...


100%|██████████| 38/38 [00:44<00:00,  1.16s/it]


LR: 2e-05
Best models saved
epoch:  4
Training...


 14%|█▎        | 46/339 [00:54<04:32,  1.08it/s]

## 7. Evaluar el modelo

Se evalúa el modelo entrenado sobre el conjunto de validación.  
Se obtienen métricas como accuracy, sensibilidad, especificidad, predicciones y etiquetas reales.

In [ ]:
# Evaluar el modelo en el conjunto de validación
acc, se, sp, preds, labels = evaluate_model(
    modelo, val_dataloader, device
)

print(f"Exactitud (Accuracy): {acc:.4f}")
print(f"Sensibilidad (Recall): {se:.4f}")
print(f"Especificidad (Specificity): {sp:.4f}")

## 8. Visualizar estadísticas de entrenamiento

Se grafican la evolución de la pérdida y la exactitud durante el entrenamiento y validación.  
Estas gráficas ayudan a identificar posibles problemas de sobreajuste o subajuste y a interpretar el desempeño del modelo.

In [ ]:
# Graficar estadísticas de entrenamiento y validación
plot_training_statistics(tr_loss, val_loss, tr_acc, val_acc)